# Phase 2 — FLAN-T5 Translation Training
**Continuous Sign Language Translation** · Phase 2 of 2

Loads the frozen `SemanticEncoder` trained in Phase 1, then fine-tunes `google/flan-t5-small` to map the continuous latent `Z` to English captions.

> ⚠️ **Run Phase 1 first** — this notebook downloads `semantic_encoder.pth` from `bdanko/continuous-sign-language-translation`.

**Runtime:** GPU (T4 or better) · **Est. time (100 clips, 2 epochs):** ~4 min


In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
%pip install -q torch datasets huggingface_hub transformers sentencepiece tqdm numpy


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Hugging Face authentication ───────────────────────────────────────────────
# Option A (recommended): Colab Secrets  →  Runtime > Manage secrets > add HF_TOKEN
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets ✓")
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("Paste your HF token: ")

import os
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)


## Model definitions

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5ForConditionalGeneration

class PartSpatialEncoder(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim), nn.GELU(),
            nn.LayerNorm(out_dim), nn.Linear(out_dim, out_dim),
        )
    def forward(self, x): return self.net(x)

class MultiStreamSemanticEncoder(nn.Module):
    def __init__(self, d_model=384, latent_dim=512):
        super().__init__()
        part_dim = d_model // 4
        self.body_encoder = PartSpatialEncoder(33*3*2, part_dim)
        self.face_encoder = PartSpatialEncoder(15*3*2, part_dim)
        self.lhand_encoder = PartSpatialEncoder(21*3*2, part_dim)
        self.rhand_encoder = PartSpatialEncoder(21*3*2, part_dim)
        self.fusion = nn.Sequential(nn.Linear(part_dim * 4, d_model), nn.GELU(), nn.LayerNorm(d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=8, dim_feedforward=d_model * 4, dropout=0.1, batch_first=True, activation="gelu"
        )
        self.temporal = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.downsample = nn.Conv1d(d_model, latent_dim, kernel_size=3, stride=2, padding=1)
        self.downsample2 = nn.Conv1d(latent_dim, latent_dim, kernel_size=3, stride=2, padding=1)

    def forward(self, inputs):
        body = torch.cat([inputs["body_pos"], inputs["body_vel"]], dim=-1)
        face = torch.cat([inputs["face_pos"], inputs["face_vel"]], dim=-1)
        lhand = torch.cat([inputs["lhand_pos"], inputs["lhand_vel"]], dim=-1)
        rhand = torch.cat([inputs["rhand_pos"], inputs["rhand_vel"]], dim=-1)
        b_feat = self.body_encoder(body)
        f_feat = self.face_encoder(face)
        l_feat = self.lhand_encoder(lhand)
        r_feat = self.rhand_encoder(rhand)
        fused = self.fusion(torch.cat([b_feat, f_feat, l_feat, r_feat], dim=-1))
        temp_out = self.temporal(fused)
        x = temp_out.transpose(1, 2)
        x = F.gelu(self.downsample(x))
        x = F.gelu(self.downsample2(x))
        return x.transpose(1, 2)

class LatentToT5Adapter(nn.Module):
    def __init__(self, latent_dim=512, t5_dim=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, t5_dim), nn.GELU(), nn.LayerNorm(t5_dim),
            nn.Dropout(dropout), nn.Linear(t5_dim, t5_dim),
        )
    def forward(self, z): return self.net(z)

class SignToTextModel(nn.Module):
    def __init__(self, encoder, latent_dim=512, t5_name="google/flan-t5-small"):
        super().__init__()
        self.encoder = encoder
        self.adapter = LatentToT5Adapter(latent_dim=latent_dim, t5_dim=512)
        self.t5 = T5ForConditionalGeneration.from_pretrained(t5_name)

    def forward(self, batch_inputs, labels=None):
        z = self.encoder(batch_inputs)
        z = self.adapter(z)
        return self.t5(inputs_embeds=z, labels=labels)

    def generate(self, batch_inputs, **kwargs):
        z = self.encoder(batch_inputs)
        z = self.adapter(z)
        return self.t5.generate(inputs_embeds=z, **kwargs)


## Dataset — streaming with on-the-fly feature engineering

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import IterableDataset

FACE_LANDMARK_IDXS = [70, 105, 336, 300, 33, 133, 362, 263, 4, 61, 291, 13, 14, 17, 0]
_POSE_SLICE = slice(0, 33)
_FACE_SLICE = slice(33, 501)
_LHAND_SLICE = slice(501, 522)
_RHAND_SLICE = slice(522, 543)
T_WINDOW, T_STRIDE = 60, 30

def engineer_features_multistream(raw: np.ndarray):
    T = raw.shape[0]
    if T < 2: return None

    pose = raw[:, _POSE_SLICE, :]
    face = raw[:, _FACE_SLICE, :][:, FACE_LANDMARK_IDXS, :]
    lhand = raw[:, _LHAND_SLICE, :]
    rhand = raw[:, _RHAND_SLICE, :]

    shoulder_center = (pose[:, 11, :] + pose[:, 12, :]) / 2.0
    pose_norm = pose - shoulder_center[:, None, :]
    
    face_center = face.mean(axis=1, keepdims=True)
    face_norm = face - face_center
    
    lhand_norm = lhand - lhand[:, 0:1, :]
    rhand_norm = rhand - rhand[:, 0:1, :]

    def calc_vel(x):
        vel = np.zeros_like(x)
        vel[1:] = x[1:] - x[:-1]
        return vel

    return {
        "body_pos": torch.from_numpy(pose_norm.reshape(T, -1).astype(np.float32)),
        "face_pos": torch.from_numpy(face_norm.reshape(T, -1).astype(np.float32)),
        "lhand_pos": torch.from_numpy(lhand_norm.reshape(T, -1).astype(np.float32)),
        "rhand_pos": torch.from_numpy(rhand_norm.reshape(T, -1).astype(np.float32)),
        "body_vel": torch.from_numpy(calc_vel(pose_norm).reshape(T, -1).astype(np.float32)),
        "face_vel": torch.from_numpy(calc_vel(face_norm).reshape(T, -1).astype(np.float32)),
        "lhand_vel": torch.from_numpy(calc_vel(lhand_norm).reshape(T, -1).astype(np.float32)),
        "rhand_vel": torch.from_numpy(calc_vel(rhand_norm).reshape(T, -1).astype(np.float32))
    }

def sliding_windows_dict(feat_dict, window=T_WINDOW, stride=T_STRIDE):
    T = feat_dict["body_pos"].shape[0]
    if T < window:
        yield {k: F.pad(v, (0, 0, 0, window - T)) for k, v in feat_dict.items()}
    else:
        start = 0
        while start + window <= T:
            yield {k: v[start:start + window] for k, v in feat_dict.items()}
            start += stride

class RealSignLanguageDataset(IterableDataset):
    def __init__(self, split="train", max_samples=None, repo_id="bdanko/how2sign-landmarks-front-raw-parquet"):
        self.split, self.max_samples, self.repo_id = split, max_samples, repo_id

    def __iter__(self):
        from datasets import load_dataset
        ds = load_dataset(self.repo_id, split=self.split, streaming=True)
        count = 0
        for sample in ds:
            if self.max_samples and count >= self.max_samples:
                break
            raw = np.frombuffer(sample["features"], dtype=np.float32).reshape(sample["shape"])
            feat_dict = engineer_features_multistream(raw)
            if feat_dict is None:
                continue
            for chunk in sliding_windows_dict(feat_dict):
                yield chunk, sample.get("sentence", "")
            count += 1


## ⚙️ Configuration


In [ ]:
# ── Hyperparameters ─────────────────────────────────────────────────────────────
ENCODER_REPO        = "bdanko/continuous-sign-language-translation"
TRANS_REPO          = "bdanko/continuous-sign-language-translation"
SPLIT               = "train"
DEBUG_MAX_SAMPLES   = 100
FULL_MAX_SAMPLES    = None
MAX_SAMPLES         = DEBUG_MAX_SAMPLES  # Switch to FULL_MAX_SAMPLES for real run
BATCH_SIZE          = 8
EPOCHS              = 3
WARMUP_EPOCHS       = 1
CKPT_DIR            = "/tmp/phase2_ckpt"


## Load & freeze the Semantic Encoder

In [ ]:
import torch
from huggingface_hub import hf_hub_download

encoder = MultiStreamSemanticEncoder().to(device)
weights = hf_hub_download(repo_id=ENCODER_REPO, filename="semantic_encoder.pth", token=HF_TOKEN)
encoder.load_state_dict(torch.load(weights, map_location=device))

print("Multi-Stream Semantic Encoder loaded ✓")


## Training loop

In [ ]:
import os, torch, torch.optim as optim
from transformers import T5Tokenizer
from tqdm.auto import tqdm
from huggingface_hub import HfApi, create_repo, upload_folder

os.makedirs(CKPT_DIR, exist_ok=True)

model = SignToTextModel(encoder).to(device)
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")

def setup_optimizer_for_stage(model, stage="warmup"):
    if stage == "warmup":
        for p in model.encoder.parameters(): p.requires_grad = False
        model.encoder.eval()
        for p in model.adapter.parameters(): p.requires_grad = True
        for p in model.t5.parameters(): p.requires_grad = True
        return torch.optim.AdamW(list(model.adapter.parameters()) + list(model.t5.parameters()), lr=1e-4)
    elif stage == "joint":
        for p in model.encoder.parameters(): p.requires_grad = True
        model.encoder.train()
        return torch.optim.AdamW([
            {"params": model.encoder.parameters(), "lr": 5e-6},
            {"params": model.adapter.parameters(), "lr": 1e-4},
            {"params": model.t5.parameters(), "lr": 5e-5}
        ])

dataset = RealSignLanguageDataset(split=SPLIT, max_samples=MAX_SAMPLES)

for epoch in range(EPOCHS):
    stage = "warmup" if epoch < WARMUP_EPOCHS else "joint"
    optimizer = setup_optimizer_for_stage(model, stage)
    print(f"Epoch {epoch+1}/{EPOCHS} [{stage.upper()}]")
    
    total_loss, count = 0.0, 0
    batch_buf, batch_labels = [], []

    for features, sentence in dataset:
        batch_buf.append(features)
        batch_labels.append(sentence)

        if len(batch_buf) < BATCH_SIZE:
            continue

        batch = {k: torch.stack([d[k] for d in batch_buf]).to(device) for k in batch_buf[0].keys()}
        labels = tokenizer(
            batch_labels, return_tensors="pt",
            padding=True, truncation=True, max_length=128
        ).input_ids.to(device)

        outputs = model(batch, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        count += 1
        batch_buf, batch_labels = [], []

    avg = total_loss / max(count, 1)
    print(f"Epoch {epoch+1}/{EPOCHS}  avg_loss={avg:.4f}")

    # Save & upload
    torch.save(model.state_dict(), f"{CKPT_DIR}/translation_model.pth")
    create_repo(TRANS_REPO, token=HF_TOKEN, exist_ok=True)
    upload_folder(
        folder_path=CKPT_DIR, repo_id=TRANS_REPO, token=HF_TOKEN,
        commit_message=f"Phase 2 epoch {epoch+1}/{EPOCHS} ({stage})",
    )
    print(f"  → checkpoint uploaded ✓")

print("Training complete.")
